# PC03 — Deep 2D CNN on Mel Spectrograms

**Classes:** 0=Speech · 1=Cough · 2=NonVerbal · 3=Other  
**Optimized for:** Macro F1 and Cough F1  
**Architecture:** 4-block 2D CNN on Mel Spectrograms (built from scratch)

## Pipeline
1. Mount Drive & install deps
2. Extract Mel Spectrograms (one file at a time, memory-safe)
3. Train Deep 2D CNN with proper validation
4. Generate predictions for test subjects
5. Download CSVs

## Folder structure expected in Google Drive
```
MyDrive/PC03-Trial-2/
├── Train/   ← 30 training .h5 files (s01–s10, 3 trials each)
└── Test/    ← 9 test .h5 files (s11–s13, 3 trials each)
```

## Cell 1 — Mount Drive & Install Dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q librosa h5py scipy scikit-learn torch torchsummary

import os
import numpy as np
import torch

# ── PATHS — edit BASE_DIR if needed ───────────────────────
BASE_DIR        = '/content/drive/MyDrive/PC03-Trial-2'
TRAIN_H5_DIR    = os.path.join(BASE_DIR, 'Train')
TEST_H5_DIR     = os.path.join(BASE_DIR, 'Test')
FEAT_TRAIN_DIR  = os.path.join(BASE_DIR, 'Features_Mel/train')
FEAT_VAL_DIR    = os.path.join(BASE_DIR, 'Features_Mel/val')
FEAT_TEST_DIR   = os.path.join(BASE_DIR, 'Features_Mel/test')
PRED_DIR        = os.path.join(BASE_DIR, 'Data-Predictions')
MODEL_PATH      = os.path.join(BASE_DIR, 'mel_cnn_model.pt')

for d in [FEAT_TRAIN_DIR, FEAT_VAL_DIR, FEAT_TEST_DIR, PRED_DIR]:
    os.makedirs(d, exist_ok=True)

# GPU check
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

train_files = sorted([f for f in os.listdir(TRAIN_H5_DIR) if f.endswith('.h5')])
test_files  = sorted([f for f in os.listdir(TEST_H5_DIR)  if f.endswith('.h5')])
print(f'Train files: {len(train_files)}')
print(f'Test files:  {len(test_files)}')

# Validation split: last 6 files (matching instructor convention)
train_list = train_files[:-6]
val_list   = train_files[-6:]
print(f'Training on {len(train_list)} files, validating on {len(val_list)} files')
print(f'Val files: {val_list}')

## Cell 2 — Constants & Mel Spectrogram Extraction

We extract a **Mel Spectrogram** for each 1-second window of audio.

Unlike MFCC (which collapses time into a single vector), the Mel Spectrogram
is a **2D matrix** (frequency × time) that preserves the temporal shape of
sounds like coughs and speech — making it ideal for 2D CNNs.

```
Window: 1 second @ 16kHz = 16,000 samples
n_mels: 64   (frequency bins)
hop:    256  (→ ~63 time frames per window)
Output per window: (1, 64, 63)  [channel, freq, time]
```

In [ ]:
import h5py
import librosa
import numpy as np
from scipy.stats import mode as scipy_mode
import time

# ── Audio / label constants ────────────────────────────────
AUDIO_RATE    = 16000
LABEL_RATE    = 10000
N_CLASSES     = 4

# ── Window parameters ──────────────────────────────────────
WINDOW_SEC    = 1.0
STRIDE_SEC    = 0.5          # 50% overlap
WINDOW_AUDIO  = int(AUDIO_RATE * WINDOW_SEC)    # 16000
STRIDE_AUDIO  = int(AUDIO_RATE * STRIDE_SEC)    # 8000
WINDOW_LABEL  = int(LABEL_RATE * WINDOW_SEC)    # 10000
STRIDE_LABEL  = int(LABEL_RATE * STRIDE_SEC)    # 5000

# ── Mel Spectrogram parameters ─────────────────────────────
N_MELS        = 64
N_FFT         = 512
HOP_LENGTH    = 256
# Expected time frames = ceil(WINDOW_AUDIO / HOP_LENGTH) = 63

def extract_mel(audio_window, sr=AUDIO_RATE):
    """
    Convert a 1-second audio window into a log Mel Spectrogram.
    Returns shape: (1, N_MELS, time_frames)  ← channel-first for PyTorch
    """
    mel = librosa.feature.melspectrogram(
        y=audio_window.astype(np.float32),
        sr=sr,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        n_mels=N_MELS
    )
    log_mel = librosa.power_to_db(mel, ref=np.max)  # log scale
    return log_mel[np.newaxis, :, :]                 # add channel dim

def process_h5_file(path, has_labels=True):
    """
    Read one .h5 file, slide a window over the audio,
    extract Mel Spectrograms, and return (X, y) or (X, None).

    X shape: (N_windows, 1, N_MELS, time_frames)
    y shape: (N_windows,)  — majority-vote label per window
    """
    with h5py.File(path, 'r') as f:
        audio     = f['audio_data'][:]
        label_arr = f['label'][:] if has_labels else None

    n_audio_wins = (len(audio) - WINDOW_AUDIO) // STRIDE_AUDIO + 1
    if has_labels:
        n_label_wins = (len(label_arr) - WINDOW_LABEL) // STRIDE_LABEL + 1
        n_wins = min(n_audio_wins, n_label_wins)
    else:
        n_wins = n_audio_wins

    X_list, y_list = [], []
    for i in range(n_wins):
        # Audio window → Mel Spectrogram
        a_start = i * STRIDE_AUDIO
        a_win   = audio[a_start : a_start + WINDOW_AUDIO]
        mel     = extract_mel(a_win)
        X_list.append(mel)

        if has_labels:
            # Label window → majority vote
            l_start = i * STRIDE_LABEL
            l_win   = label_arr[l_start : l_start + WINDOW_LABEL]
            lbl     = int(scipy_mode(l_win, keepdims=True).mode[0])
            y_list.append(lbl)

    X = np.array(X_list, dtype=np.float32)
    if has_labels:
        y    = np.array(y_list, dtype=np.int64)
        dist = np.bincount(y, minlength=N_CLASSES)
        return X, y, dist
    return X, None, None

print(f'Mel Spectrogram shape per window: (1, {N_MELS}, ~63)')
print('Extraction functions ready.')

## Cell 3 — Extract Features (Memory-Safe, Resumable)
Processes one file at a time. If Colab disconnects, re-run — already-done files are skipped.

In [ ]:
def extract_split(file_list, out_dir, has_labels=True, label=''):
    print(f'\nExtracting {label} set ({len(file_list)} files) ...')
    for fname in file_list:
        name    = os.path.splitext(fname)[0]
        out     = os.path.join(out_dir, name + '.npz')
        if os.path.exists(out):
            print(f'  SKIP {name} (already done)')
            continue
        t0   = time.time()
        path = os.path.join(TRAIN_H5_DIR if has_labels else TEST_H5_DIR, fname)
        X, y, dist = process_h5_file(path, has_labels=has_labels)
        if has_labels:
            np.savez_compressed(out, X=X, y=y)
            print(f'  {name}: {len(y)} windows | '
                  f'Speech={dist[0]} Cough={dist[1]} '
                  f'NonVerbal={dist[2]} Other={dist[3]} | {time.time()-t0:.0f}s')
        else:
            np.savez_compressed(out, X=X)
            print(f'  {name}: {len(X)} windows | {time.time()-t0:.0f}s')

# Training files
extract_split(train_list, FEAT_TRAIN_DIR, has_labels=True,  label='Train')
# Validation files (last 6 — subjects s09 & s10)
extract_split(val_list,   FEAT_VAL_DIR,   has_labels=True,  label='Validation')
# Test files
extract_split(test_files, FEAT_TEST_DIR,  has_labels=False, label='Test')

print('\nFeature extraction complete!')

## Cell 4 — Model Definition

### Architecture: Deep 2D CNN

```
Input: (B, 1, 64, 63)   ← batch × channel × freq × time
│
├─ Block 1: Conv2d(1→32,  3×3) → BN → ReLU → MaxPool(2×2)  → (B, 32, 32, 31)
├─ Block 2: Conv2d(32→64, 3×3) → BN → ReLU → MaxPool(2×2)  → (B, 64, 16, 15)
├─ Block 3: Conv2d(64→128,3×3) → BN → ReLU → MaxPool(2×2)  → (B,128,  8,  7)
├─ Block 4: Conv2d(128→128,3×3)→ BN → ReLU → AdaptAvgPool  → (B,128,  1,  1)
│
└─ Classifier: Flatten → Linear(128→64) → ReLU → Dropout(0.4) → Linear(64→4)
```

**Why this is better than PC02:**
- PC02 used 4 filters max — we use up to 128
- PC02 had no BatchNorm — we add it after every conv
- PC02 used flat MaxPool — we use AdaptiveAvgPool at the end
- PC02 classifier had 16 hidden units — we use 64 with Dropout

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.metrics import classification_report, f1_score
import glob

class ConvBlock(nn.Module):
    """Conv2d → BatchNorm → ReLU → Pooling"""
    def __init__(self, in_ch, out_ch, pool='max'):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1)
        self.bn   = nn.BatchNorm2d(out_ch)
        self.relu = nn.ReLU()
        if pool == 'max':
            self.pool = nn.MaxPool2d(2)
        else:  # adaptive — used in last block
            self.pool = nn.AdaptiveAvgPool2d((1, 1))

    def forward(self, x):
        return self.pool(self.relu(self.bn(self.conv(x))))


class MelCNN(nn.Module):
    """
    Deep 2D CNN for Mel Spectrogram classification.
    Built from scratch — no pretrained weights.

    Input:  (B, 1, n_mels, time_frames)
    Output: (B, n_classes)
    """
    def __init__(self, n_classes=4):
        super().__init__()

        # 4 convolutional blocks with increasing filter counts
        self.block1 = ConvBlock(1,   32,  pool='max')   # learns edges/textures
        self.block2 = ConvBlock(32,  64,  pool='max')   # learns patterns
        self.block3 = ConvBlock(64,  128, pool='max')   # learns complex features
        self.block4 = ConvBlock(128, 128, pool='adaptive')  # global summary

        # Classifier head
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.4),        # prevents memorization
            nn.Linear(64, n_classes)
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        return self.classifier(x)


class MelDataset(Dataset):
    def __init__(self, X, y=None):
        self.X = torch.from_numpy(X).float()
        self.y = torch.from_numpy(y).long() if y is not None else None
    def __len__(self): return len(self.X)
    def __getitem__(self, i):
        return (self.X[i], self.y[i]) if self.y is not None else self.X[i]


# Print model summary
from torchsummary import summary
model = MelCNN(n_classes=N_CLASSES).to(DEVICE)
print('Model Architecture:')
summary(model, input_size=(1, N_MELS, 63))

## Cell 5 — Load Features & Normalise

We normalise the Mel Spectrograms using training set statistics (mean/std per frequency bin).
This is important because different frequency bins have very different energy ranges.

In [ ]:
def load_npz_dir(directory):
    """Load all .npz files from a directory, return (X, y) or (X, None)."""
    files = sorted(glob.glob(os.path.join(directory, '*.npz')))
    X_list, y_list = [], []
    for path in files:
        d = np.load(path)
        X_list.append(d['X'])
        if 'y' in d:
            y_list.append(d['y'])
            dist = np.bincount(d['y'], minlength=N_CLASSES)
            print(f"  {os.path.basename(path)}: {len(d['y'])} windows | "
                  f"Speech={dist[0]} Cough={dist[1]} NonVerbal={dist[2]} Other={dist[3]}")
        else:
            print(f"  {os.path.basename(path)}: {len(d['X'])} windows")
    X = np.concatenate(X_list, axis=0)
    y = np.concatenate(y_list, axis=0) if y_list else None
    return X, y

print('Loading training features ...')
X_train, y_train = load_npz_dir(FEAT_TRAIN_DIR)
print(f'\nTrain: {X_train.shape}  Label dist: {np.bincount(y_train, minlength=N_CLASSES)}')

print('\nLoading validation features ...')
X_val, y_val = load_npz_dir(FEAT_VAL_DIR)
print(f'Val: {X_val.shape}  Label dist: {np.bincount(y_val, minlength=N_CLASSES)}')

# Normalise using training set statistics
# Shape: (N, 1, 64, time) → compute mean/std over N and time, per freq bin
mel_mean = X_train.mean(axis=(0, 3), keepdims=True)  # (1, 1, 64, 1)
mel_std  = X_train.std( axis=(0, 3), keepdims=True) + 1e-8

X_train_n = (X_train - mel_mean) / mel_std
X_val_n   = (X_val   - mel_mean) / mel_std

# Save normalisation stats for test-time use
np.savez(os.path.join(BASE_DIR, 'mel_norm_stats.npz'), mean=mel_mean, std=mel_std)
print(f'\nNormalisation stats saved.')
print(f'X_train_n: {X_train_n.shape}, X_val_n: {X_val_n.shape}')

## Cell 6 — Train the Model

**Key training decisions:**
- `WeightedRandomSampler`: oversamples minority classes in each batch (better than repeating)
- `WeightedCrossEntropy`: Cough=4×, NonVerbal=2×, Speech=1.5×
- `AdamW` optimizer with cosine LR schedule
- Best model saved on validation Macro F1 (not just accuracy)

In [ ]:
BATCH_SIZE = 128
EPOCHS     = 50
LR         = 1e-3

# Class weights — Cough is hardest and most important
class_weights = torch.tensor([1.5, 4.0, 2.0, 1.0], dtype=torch.float32).to(DEVICE)

# WeightedRandomSampler — ensures each batch has balanced class representation
sample_weights = np.array([class_weights[y].item() for y in y_train])
sampler = WeightedRandomSampler(
    weights=torch.from_numpy(sample_weights).float(),
    num_samples=len(sample_weights),
    replacement=True
)

train_loader = DataLoader(
    MelDataset(X_train_n, y_train),
    batch_size=BATCH_SIZE,
    sampler=sampler,
    num_workers=2
)
val_loader = DataLoader(
    MelDataset(X_val_n, y_val),
    batch_size=256,
    shuffle=False,
    num_workers=2
)

# Fresh model
model     = MelCNN(n_classes=N_CLASSES).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_macro_f1 = 0.0
print(f'Training for {EPOCHS} epochs on {DEVICE} ...')
print(f'{"Epoch":>6} {"TrainLoss":>10} {"TrainAcc":>10} {"ValMacroF1":>12} {"ValCoughF1":>11}')
print('-' * 55)

for epoch in range(1, EPOCHS + 1):
    # ── Train ──────────────────────────────────
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        logits = model(xb)
        loss   = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(yb)
        correct    += (logits.argmax(1) == yb).sum().item()
        total      += len(yb)
    scheduler.step()
    train_loss = total_loss / total
    train_acc  = 100 * correct / total

    # ── Validate ───────────────────────────────
    model.eval()
    val_preds, val_true = [], []
    with torch.no_grad():
        for xb, yb in val_loader:
            val_preds.append(model(xb.to(DEVICE)).argmax(1).cpu().numpy())
            val_true.append(yb.numpy())
    val_preds = np.concatenate(val_preds)
    val_true  = np.concatenate(val_true)

    macro_f1 = f1_score(val_true, val_preds, average='macro',    zero_division=0)
    cough_f1 = f1_score(val_true, val_preds, average=None,       zero_division=0)[1]

    flag = ''
    if macro_f1 > best_macro_f1:
        best_macro_f1 = macro_f1
        torch.save(model.state_dict(), MODEL_PATH)
        flag = ' ← best'

    print(f'{epoch:>6}  {train_loss:>10.4f}  {train_acc:>9.1f}%  '
          f'{macro_f1:>11.4f}  {cough_f1:>10.4f}{flag}')

print(f'\nBest Val Macro F1: {best_macro_f1:.4f}')
print(f'Model saved to {MODEL_PATH}')

## Cell 7 — Full Validation Report

In [ ]:
# Load best model and evaluate
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()

val_preds, val_true = [], []
with torch.no_grad():
    for xb, yb in val_loader:
        val_preds.append(model(xb.to(DEVICE)).argmax(1).cpu().numpy())
        val_true.append(yb.numpy())
val_preds = np.concatenate(val_preds)
val_true  = np.concatenate(val_true)

print('=== Validation Results (subjects s09-s10, unseen during training) ===')
print(classification_report(val_true, val_preds,
                             target_names=['Speech','Cough','NonVerbal','Other']))
print(f'Macro F1:  {f1_score(val_true, val_preds, average="macro",    zero_division=0):.4f}')
print(f'Cough F1:  {f1_score(val_true, val_preds, average=None,       zero_division=0)[1]:.4f}')

## Cell 8 — Generate Predictions for Test Files

For each test file:
1. Run the model on each window → get per-window label
2. Aggregate overlapping windows by voting
3. Upsample from window-level → sample-level (10,000 Hz)
4. Write CSV matching exact expected length

In [ ]:
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()

# Load normalisation stats
norm_data = np.load(os.path.join(BASE_DIR, 'mel_norm_stats.npz'))
mel_mean  = norm_data['mean']
mel_std   = norm_data['std']

feat_test_files = sorted(glob.glob(os.path.join(FEAT_TEST_DIR, '*.npz')))
print(f'Generating predictions for {len(feat_test_files)} test files ...')
print(f'{"File":<20} {"Rows":>10} {"Speech":>8} {"Cough":>8} {"NonVerbal":>10} {"Other":>8}')
print('-' * 70)

for feat_path in feat_test_files:
    name     = os.path.splitext(os.path.basename(feat_path))[0]
    out_path = os.path.join(PRED_DIR, name + '.csv')

    # Load and normalise
    X = (np.load(feat_path)['X'] - mel_mean) / mel_std

    # Per-window predictions
    pred_wins = []
    with torch.no_grad():
        ds = MelDataset(X)
        for xb in DataLoader(ds, batch_size=256, num_workers=2):
            pred_wins.append(model(xb.to(DEVICE)).argmax(1).cpu().numpy())
    pred_wins = np.concatenate(pred_wins)   # (N_windows,)

    # Upsample: each window covers WINDOW_LABEL label samples, stride STRIDE_LABEL
    n_wins     = len(pred_wins)
    total_samp = (n_wins - 1) * STRIDE_LABEL + WINDOW_LABEL
    votes      = np.zeros(total_samp, dtype=np.float64)
    counts     = np.zeros(total_samp, dtype=np.int32)
    for i, p in enumerate(pred_wins):
        s = i * STRIDE_LABEL
        e = s + WINDOW_LABEL
        votes[s:e]  += p
        counts[s:e] += 1
    per_sample = np.clip(
        np.round(votes / np.maximum(counts, 1)), 0, 3
    ).astype(np.int64)

    # Match exact target length from dummy CSV
    dummy = os.path.join(PRED_DIR, name + '.csv')
    if os.path.exists(dummy):
        with open(dummy) as f:
            target = sum(1 for _ in f)
    else:
        target = len(per_sample)

    if len(per_sample) >= target:
        final = per_sample[:target]
    else:
        pad   = np.full(target - len(per_sample), per_sample[-1], dtype=np.int64)
        final = np.concatenate([per_sample, pad])

    np.savetxt(out_path, final.astype(int), fmt='%d')
    dist = np.bincount(final, minlength=N_CLASSES)
    print(f'{name:<20} {len(final):>10,} {dist[0]:>8,} {dist[1]:>8,} {dist[2]:>10,} {dist[3]:>8,}')

print('\nAll predictions written!')

## Cell 9 — Download Predictions
Downloads all 9 CSVs. Copy them into your repo's `Data-Predictions/` folder and push to Git.

In [ ]:
from google.colab import files

pred_files = sorted(glob.glob(os.path.join(PRED_DIR, '*.csv')))
print(f'Downloading {len(pred_files)} prediction files ...')
for f in pred_files:
    files.download(f)
print('Done!')
print()
print('Next steps:')
print('1. Copy CSVs into Data-Predictions/ in your repo')
print('2. Run: python3 test_format.py')
print('3. git add Data-Predictions/ mel_cnn_model.pt')
print('4. git commit -m "Trial 2 - Deep 2D CNN on Mel Spectrograms"')
print('5. git push')